# Codigo para sacar embedings tanto de CLIP como de DINO

In [ ]:
import torch
import torch.nn as nn
import clip
from PIL import Image
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
MODELO = "dinov2"  # Cambiar a "clip" para usar CLIP ViT-L/14

datasets_info = [
    {
        "ruta": "",
        "nombre_origen": "DigiFace",
        "out_npy": "embeddings_DINO_digiface133333.npy",
        "out_csv": "embeddings_DINO_digiface133333.csv"
    }
]

LIMITE_IMAGENES_POR_DATASET = 50000

# ==========================================
# 2. CARGA DEL MODELO + DataParallel
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if MODELO == "dinov2":
    import torchvision.transforms as T
    print(f"Cargando DINOv2 ViT-L/14 reg...")
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14_reg')

    preprocess = T.Compose([
        T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])

    def extraer_features(images):
        out = model(images)
        if not isinstance(out, torch.Tensor):
            out = out[0]
        return out  # (B, 1024)

else:
    print(f"Cargando CLIP ViT-L/14...")
    clip_model, preprocess = clip.load("ViT-L/14", device="cpu")
    model = clip_model.visual

    def extraer_features(images):
        out = model(images)
        if not isinstance(out, torch.Tensor):
            out = out[0]
        return out  # (B, 768)

# ← DataParallel aquí, antes de mover a device
if torch.cuda.device_count() > 1:
    print(f" Usando {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(device)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ==========================================
# 3. DATASET
# ==========================================
class FaceDataset(Dataset):
    def __init__(self, root_dir, limit=None, transform=None):
        self.samples   = []
        self.transform = transform
        count = 0
        EXTENSIONES_VALIDAS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

        for identity in sorted(os.listdir(root_dir)):
            id_path = os.path.join(root_dir, identity)
            if os.path.isdir(id_path):
                for img_name in os.listdir(id_path):
                    if limit and count >= limit:
                        break
                    ext = os.path.splitext(img_name)[1].lower()
                    if ext not in EXTENSIONES_VALIDAS:
                        continue
                    self.samples.append((os.path.join(id_path, img_name), identity))
                    count += 1
            if limit and count >= limit:
                break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, identity = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[SKIP] Imagen corrupta: {img_path} → {e}")
            return None, None
        if self.transform:
            image = self.transform(image)
        return image, identity

# ==========================================
# 4.EXTRACCIÓN
# ==========================================
def extraer_y_guardar(info):
    print(f"\n--- Procesando: {info['nombre_origen']} con {MODELO.upper()} ---")
    dataset = FaceDataset(info['ruta'], limit=LIMITE_IMAGENES_POR_DATASET, transform=preprocess)
    loader  = DataLoader(dataset, batch_size=128, shuffle=False,  # ← batch mayor con 2 GPUs
                         num_workers=2, pin_memory=True)

    embeddings_list = []
    identidades     = []

    with torch.no_grad():
        for batch_imgs, batch_ids in tqdm(loader, desc=f"Extrayendo {info['nombre_origen']}"):
            imgs     = batch_imgs.to(device, non_blocking=True)
            features = extraer_features(imgs)
            features = features / features.norm(dim=-1, keepdim=True)  # L2
            embeddings_list.append(features.cpu().numpy())
            identidades.extend(batch_ids)

    rutas           = [s[0] for s in dataset.samples]
    rutas_relativas = [os.path.relpath(s[0], info['ruta']) for s in dataset.samples]

    embeddings_array = np.vstack(embeddings_list)
    assert len(embeddings_array) == len(identidades) == len(rutas), \
        f"¡Desajuste! emb:{len(embeddings_array)} ids:{len(identidades)} rutas:{len(rutas)}"

    np.save(info['out_npy'], embeddings_array)

    df = pd.DataFrame({
        'img_path':          rutas,
        'img_path_relativo': rutas_relativas,
        'identity':          identidades,
        'origen':            info['nombre_origen'],
        'modelo':            MODELO
    })
    df.to_csv(info['out_csv'], index=False)

    print(f" Guardado: {info['out_npy']} {embeddings_array.shape} | {info['out_csv']}")
    print(f"   Total imágenes procesadas: {len(df)}")
    print(f"   VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ==========================================
# 5. EJECUTAR
# ==========================================
if __name__ == "__main__":
    for ds_info in datasets_info:
        if os.path.exists(ds_info['ruta']):
            extraer_y_guardar(ds_info)
        else:
            print(f"\n[ERROR] Ruta no encontrada: {ds_info['ruta']}")
    print("\n¡PROCESO TOTAL COMPLETADO!")

# Extraccion de los embeddings de RESNET-50

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from torchvision import models, transforms

# ==========================================
# 1. CONFIGURACIÓN DE DATASETS (MULTIPLE)
# ==========================================
DIRECTORIO_OUT = ""  # Cambia a tu ruta de salida
LIMITE_IMAGENES = 250000
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


DATASETS_A_PROCESAR = [
    {
        "ruta_raiz": "",
        "origen": "IDiffFace",
        "modelo": "resnet50_base",
        "nombre_archivo": "idiff_face"
    }
]

# ==========================================
# 2. RESNET-50 BASE (ImageNet)
# ==========================================
print(f"Cargando ResNet-50 Base (Pesos ImageNet)...")
# Usamos el preentrenado estándar de ImageNet
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1) 

model.fc = nn.Identity() 
model.to(DEVICE)
model.eval()

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 3. DATASET
# ==========================================
class MultiExtractionDataset(Dataset):
    def __init__(self, root_dir, transform, limite=50000):
        self.samples = []
        self.transform = transform
        extensiones = {'.jpg', '.jpeg', '.png'}
        self.root_dir = root_dir

        for identity in sorted(os.listdir(root_dir)):
            if len(self.samples) >= limite: break
                
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path): continue
                
            for img_name in sorted(os.listdir(id_path)):
                if len(self.samples) >= limite: break
                    
                if os.path.splitext(img_name)[1].lower() in extensiones:
                    ruta_absoluta = os.path.join(id_path, img_name)
                    ruta_relativa = os.path.relpath(ruta_absoluta, root_dir)
                    
                    self.samples.append((ruta_absoluta, ruta_relativa, identity))

    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        ruta_abs, ruta_rel, identity = self.samples[idx]
        img = self.transform(Image.open(ruta_abs).convert("RGB"))
        return img, ruta_abs, ruta_rel, identity

# ==========================================
# 4. BUCLE DE EXTRACCIÓN
# ==========================================
for config in DATASETS_A_PROCESAR:
    print(f"\n{'='*60}")
    print(f" INICIANDO EXTRACCIÓN: {config['origen']}")
    print(f"{'='*60}")
    
    dataset = MultiExtractionDataset(config["ruta_raiz"], val_transform, limite=LIMITE_IMAGENES)
    
    # Si la carpeta no existe o no tiene imágenes, saltamos al siguiente
    if len(dataset) == 0:
        print(f" No se encontraron imágenes en {config['ruta_raiz']}. Saltando...")
        continue
        
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    embs, absolutas, relativas, identidades = [], [], [], []

    print(f"Procesando {len(dataset)} imágenes...")

    with torch.no_grad():
        for imgs, r_abs, r_rel, ids in tqdm(loader, desc=f"Extrayendo {config['nombre_archivo']}"):
            imgs = imgs.to(DEVICE)
            with torch.amp.autocast('cuda'):
                features = model(imgs)
            # Normalización L2
            features = F.normalize(features.float(), dim=-1)
            embs.append(features.cpu().numpy())
            absolutas.extend(r_abs)
            relativas.extend(r_rel)
            identidades.extend(ids)

    embs_array = np.vstack(embs)

    npy_path = os.path.join(DIRECTORIO_OUT, f"embeddings_{config['nombre_archivo']}_{config['modelo']}.npy")
    csv_path = os.path.join(DIRECTORIO_OUT, f"embeddings_{config['nombre_archivo']}_{config['modelo']}.csv")

    np.save(npy_path, embs_array)
    
    # Guardar .csv con tu formato exacto
    df = pd.DataFrame({
        'img_path':          absolutas,
        'img_path_relativo': relativas,
        'identity':          identidades,
        'origen':            config['origen'],
        'modelo':            config['modelo']
    })
    df.to_csv(csv_path, index=False)

    print(f" Completado {config['origen']}:")
    print(f"    NPY guardado en: {npy_path} | Shape: {embs_array.shape}")
    print(f"   CSV guardado en: {csv_path}")


# Codigo para sacar PCA de embeddings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
DIRECTORIO_OUT     = r"C:\Users\Usuario\Documents\TFG-INFo\PCA"
os.makedirs(DIRECTORIO_OUT, exist_ok=True)

identidades_a_mostrar = [0, 1, 2, 3, 4, 5,6,7,8,9]  # 10 identidades

modelos = [
    {
        "nombre":       "CLIP base",
        "nombre_arch":  "clip_base",
        "ruta_npy":     r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\embeddings_casia-webface.npy",
        "ruta_csv":     r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\embeddings_casia-webface.csv",
        "titulo":       "CLIP ViT-L/14 - base",
    },
    {
        "nombre":       "CLIP + LoRA",
        "nombre_arch":  "clip_lora",
        "ruta_npy":     r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\Modelo_lora_DIGI\embeddings_casia_clip_lora_ce.npy",
        "ruta_csv":     r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\Modelo_lora_DIGI\embeddings_casia_clip_lora_ce.csv",
        "titulo":       "CLIP ViT-L/14 + LoRA",
    },
    {
        "nombre":       "DINOv2 base",
        "nombre_arch":  "dino_base",
        "ruta_npy":     r"C:\Users\Usuario\Documents\TFG-INFo\DINOv2-vitl14_reg\embeddings_casia_dinov2.npy",
        "ruta_csv":     r"C:\Users\Usuario\Documents\TFG-INFo\DINOv2-vitl14_reg\embeddings_casia_dinov2.csv",
        "titulo":       "DINOv2 ViT-L/14 - base",
    },
    {
        "nombre":       "DINOv2 + LoRA",
        "nombre_arch":  "dino_lora",
        "ruta_npy":     r"C:\Users\Usuario\Documents\TFG-INFo\DINOv2-vitl14_reg\modelo_lora_IDIFF\embeddings_casia_dino_lora_ce.npy",
        "ruta_csv":     r"C:\Users\Usuario\Documents\TFG-INFo\DINOv2-vitl14_reg\modelo_lora_IDIFF\embeddings_casia_dino_lora_ce.csv",
        "titulo":       "DINOv2 ViT-L/14 + LoRA",
    },
]

# ==========================================
# 2. FUNCIÓN DE PCA
# ==========================================
def generar_pca(modelo_info, identidades, directorio_out):
    print(f"\nProcesando: {modelo_info['nombre']}...")
    embeddings = np.load(modelo_info['ruta_npy'])
    df         = pd.read_csv(modelo_info['ruta_csv'])
    assert len(embeddings) == len(df), \
        f"Desajuste en {modelo_info['nombre']}: emb={len(embeddings)} csv={len(df)}"
    # Normalizar tipo de identidad
    df['identity'] = df['identity'].astype(int)
    # Filtrar identidades
    mascara = df['identity'].isin(identidades)
    X = embeddings[mascara]
    y = df[mascara]['identity'].values
    print(f"  Muestras tras filtrado: {len(X)} | Identidades: {len(np.unique(y))}")
    X_scaled = StandardScaler().fit_transform(X)
    pca      = PCA(n_components=2, random_state=42)
    X_pca    = pca.fit_transform(X_scaled)

    varianza_pc1 = pca.explained_variance_ratio_[0] * 100
    varianza_pc2 = pca.explained_variance_ratio_[1] * 100

    print(f"  PC1: {varianza_pc1:.2f}% | PC2: {varianza_pc2:.2f}%")

    
    df_pca = pd.DataFrame({
        'PC1':       X_pca[:, 0],
        'PC2':       X_pca[:, 1],
        'Identidad': y.astype(str)
    })
    # Gráfica
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.set_theme(style="whitegrid")
    sns.scatterplot(
        data=df_pca,
        x='PC1', y='PC2',
        hue='Identidad',
        palette='tab10',
        s=80,
        alpha=0.8,
        edgecolor='k',
        linewidth=0.4,
        ax=ax
    )
    ax.set_title(modelo_info['titulo'], fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel(f'PC1 ({varianza_pc1:.1f}%)', fontsize=11)
    ax.set_ylabel(f'PC2 ({varianza_pc2:.1f}%)', fontsize=11)
    ax.legend(title='Identidad', bbox_to_anchor=(1.05, 1),
              loc='upper left', frameon=True, fontsize=9)
    plt.tight_layout()
    ruta_salida = os.path.join(directorio_out, f"pca_{modelo_info['nombre_arch']}.png")
    plt.savefig(ruta_salida, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Guardado: {ruta_salida}")

# ==========================================
# 3. GENERAR LOS 4 PCA
# ==========================================
for modelo in modelos:
    generar_pca(modelo, identidades_a_mostrar, DIRECTORIO_OUT)

print("\n Los 4 PCA han sido generados correctamente")

# Codigo para calcular el EER y la curva de ROC. Y grafica de distribuccion de genuinos e impostores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import os

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
DIRECTORIO_OUT = ""
N_PARES        = 2000
LIMITE_PARES   = 3000  # <-- los pares SIEMPRE se generan sobre las primeras 3000 imágenes del base

ruta_npy_base = r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\embeddings_casia-webface.npy"
ruta_csv_base = r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\embeddings_casia-webface.csv"

ruta_npy_lora = r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\Modelo_lora_DIGI\embeddings_casia_clip_lora_ce.npy"
ruta_csv_lora = r"C:\Users\Usuario\Documents\TFG-INFo\CLIP-VITL14\Modelo_lora_DIGI\embeddings_casia_clip_lora_ce.csv"

# ==========================================
# 2. CARGAR EMBEDDINGS
# ==========================================
print("Cargando embeddings...")
emb_base = np.load(ruta_npy_base)
df_base  = pd.read_csv(ruta_csv_base)

emb_lora = np.load(ruta_npy_lora)
df_lora  = pd.read_csv(ruta_csv_lora)

print(f"CLIP base total: {len(emb_base)} | CLIP LoRA: {len(emb_lora)}")

# Verificar que el LoRA tiene al menos LIMITE_PARES imágenes
assert len(emb_lora) >= LIMITE_PARES, \
    f" El LoRA solo tiene {len(emb_lora)} embeddings, necesita al menos {LIMITE_PARES}."

# Verificar que el orden coincide en las primeras LIMITE_PARES imágenes
assert np.array_equal(
    df_base['identity'].values[:LIMITE_PARES],
    df_lora['identity'].values[:LIMITE_PARES]
), " Las imágenes no están en el mismo orden en ambos CSV."

print(f" Orden verificado en las primeras {LIMITE_PARES} imágenes")

# ==========================================
# 3. GENERAR PARES FIJOS SOBRE LAS PRIMERAS
#    LIMITE_PARES IMÁGENES DEL BASE
#     índices siempre en [0, LIMITE_PARES)
#     cualquier LoRA con ≥ LIMITE_PARES imágenes los cubre
# ==========================================
print(f"\nGenerando {N_PARES} pares genuinos y {N_PARES} impostores sobre las primeras {LIMITE_PARES} imágenes...")

etiquetas_fijas   = df_base['identity'].values[:LIMITE_PARES]
identidades_unicas = np.unique(etiquetas_fijas)
indices_por_id     = {ide: np.where(etiquetas_fijas == ide)[0] for ide in identidades_unicas}
ids_validas        = [ide for ide in identidades_unicas if len(indices_por_id[ide]) >= 2]

print(f"Identidades con ≥2 imágenes en las primeras {LIMITE_PARES}: {len(ids_validas)}")

rng       = np.random.default_rng(42)
pares_gen = []
pares_imp = []

while len(pares_gen) < N_PARES:
    ide  = rng.choice(ids_validas)
    i, j = rng.choice(indices_por_id[ide], size=2, replace=False)
    pares_gen.append((int(i), int(j)))

while len(pares_imp) < N_PARES:
    id1, id2 = rng.choice(ids_validas, size=2, replace=False)
    i = rng.choice(indices_por_id[id1])
    j = rng.choice(indices_por_id[id2])
    pares_imp.append((int(i), int(j)))

idx_max = max(max(p) for p in pares_gen + pares_imp)
print(f" Pares generados - índice máximo usado: {idx_max} (límite: {LIMITE_PARES - 1})")

# ==========================================
# 4. RECORTAR EMBEDDINGS A LIMITE_PARES
# ==========================================
emb_base_eval = emb_base[:LIMITE_PARES]
emb_lora_eval = emb_lora[:LIMITE_PARES]

# ==========================================
# 5. FUNCIÓN DE EVALUACIÓN
# ==========================================
def evaluar_con_pares(embeddings, pares_gen, pares_imp, nombre):
    scores_gen = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in pares_gen])
    scores_imp = np.array([np.dot(embeddings[i], embeddings[j]) for i, j in pares_imp])

    print(f"\n{'='*50}")
    print(f"Evaluando: {nombre}")
    print(f"{'='*50}")
    print(f"Similitud media genuinos:   {scores_gen.mean():.4f} ± {scores_gen.std():.4f}")
    print(f"Similitud media impostores: {scores_imp.mean():.4f} ± {scores_imp.std():.4f}")
    print(f"Separación (gap):           {scores_gen.mean() - scores_imp.mean():.4f}")

    scores = np.concatenate([scores_gen, scores_imp])
    y_true = np.concatenate([np.ones(len(pares_gen)), np.zeros(len(pares_imp))])

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    fnr     = 1 - tpr
    idx_eer = np.nanargmin(np.abs(fnr - fpr))
    eer     = fpr[idx_eer]
    umbral  = thresholds[idx_eer]

    print(f"AUC:           {roc_auc:.4f}")
    print(f"EER:           {eer * 100:.2f}%")
    print(f"Umbral óptimo: {umbral:.4f}")

    return {
        'nombre':     nombre,
        'scores_gen': scores_gen,
        'scores_imp': scores_imp,
        'fpr':        fpr,
        'tpr':        tpr,
        'roc_auc':    roc_auc,
        'eer':        eer,
        'umbral':     umbral,
        'idx_eer':    idx_eer
    }

# ==========================================
# 6. EVALUAR AMBOS MODELOS CON LOS MISMOS PARES
# ==========================================
resultados_base = evaluar_con_pares(emb_base_eval, pares_gen, pares_imp, "CLIP base")
resultados_lora = evaluar_con_pares(emb_lora_eval, pares_gen, pares_imp, "CLIP + LoRA (DigiFace)")

# ==========================================
# 7. GRÁFICAS
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Comparativa CLIP base vs CLIP + LoRA - CASIA-WebFace\n"
             f"(pares fijos sobre las primeras {LIMITE_PARES} imágenes, semilla 42)",
             fontsize=14, fontweight='bold')

for idx, r in enumerate([resultados_base, resultados_lora]):
    color = ['steelblue', 'darkorange'][idx]

    axes[0, idx].plot(r['fpr'], r['tpr'], color=color, lw=2, label=f"ROC (AUC = {r['roc_auc']:.4f})")
    axes[0, idx].plot([0,1], [0,1], 'navy', lw=1, linestyle='--')
    axes[0, idx].plot(r['fpr'][r['idx_eer']], r['tpr'][r['idx_eer']], 'ro',
                      markersize=8, label=f"EER = {r['eer']*100:.2f}%")
    axes[0, idx].set_title(f"Curva ROC -- {r['nombre']}")
    axes[0, idx].set_xlabel("False Positive Rate")
    axes[0, idx].set_ylabel("True Positive Rate")
    axes[0, idx].legend(loc="lower right")
    axes[0, idx].grid(True, alpha=0.3)

    axes[1, idx].hist(r['scores_imp'], bins=50, alpha=0.6, color='red',   density=True, label='Impostores')
    axes[1, idx].hist(r['scores_gen'], bins=50, alpha=0.6, color='green', density=True, label='Genuinos')
    axes[1, idx].axvline(r['umbral'], color='black', linestyle='--', lw=2,
                         label=f"Umbral ({r['umbral']:.2f})")
    axes[1, idx].set_title(f"Distribución - {r['nombre']}")
    axes[1, idx].set_xlabel("Similitud coseno")
    axes[1, idx].set_ylabel("Densidad")
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
ruta_grafica = os.path.join(DIRECTORIO_OUT, "comparativa_base_vs_lora.png")
plt.savefig(ruta_grafica, dpi=300, bbox_inches='tight')
plt.show()
print(f" Gráfica guardada: {ruta_grafica}")

# ==========================================
# 8. TABLA RESUMEN
# ==========================================
print(f"\n{'='*50}")
print(f"{'RESUMEN COMPARATIVA':^50}")
print(f"{'='*50}")
print(f"{'Modelo':<20} {'EER':>8} {'AUC':>8} {'Gap':>8}")
print(f"{'-'*50}")
for r in [resultados_base, resultados_lora]:
    gap = r['scores_gen'].mean() - r['scores_imp'].mean()
    print(f"{r['nombre']:<20} {r['eer']*100:>7.2f}% {r['roc_auc']:>8.4f} {gap:>8.4f}")
print(f"{'='*50}")
print(f"\n  Evaluación fija sobre {LIMITE_PARES} imágenes  {N_PARES}×2 pares · semilla 42")

# GRAD-CAM

In [ ]:
import torch
import torch.nn as nn
import clip
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from peft import PeftModel, LoraConfig, get_peft_model
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import torchvision.transforms as T
import os

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Ejecutando en: {device}")

CHECKPOINT_CLIP  = ""
CHECKPOINT_DINO  = ""

# Imágenes de CASIA-WebFace a visualizar 
IMAGENES = [
    r"C:\Users\Usuario\Documents\TFG-INFo\casiaWebface\casia-webface\000000\00000001.jpg",
    r"C:\Users\Usuario\Documents\TFG-INFo\casiaWebface\casia-webface\000001\00000016.jpg",
    r"C:\Users\Usuario\Documents\TFG-INFo\casiaWebface\casia-webface\000002\00000272.jpg",
    r"C:\Users\Usuario\Documents\TFG-INFo\casiaWebface\casia-webface\000003\00000341.jpg",
]

DIRECTORIO_OUT = r"C:\Users\Usuario\Documents\TFG-INFo\gradcam"
os.makedirs(DIRECTORIO_OUT, exist_ok=True)

# Normalización CLIP
MEAN_CLIP = torch.tensor([0.48145466, 0.4578275,  0.40821073]).view(3,1,1).to(device)
STD_CLIP  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1).to(device)

# Normalización DINOv2
MEAN_DINO = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1).to(device)
STD_DINO  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1).to(device)

# ==========================================
# 2. WRAPPERS
# ==========================================
class CLIPWrapper(nn.Module):
    def __init__(self, visual):
        super().__init__()
        self.visual = visual
    def forward(self, x):
        return self.visual(x)

class DINOWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x)

def reshape_transform_clip(tensor, height=16, width=16):
    tensor = tensor.permute(1, 0, 2)
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

def reshape_transform_dino(tensor, height=16, width=16):
    result = tensor[:, 1:height*width+1, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

# ==========================================
# 3. CARGA DE MODELOS
# ==========================================
print("Cargando CLIP base...")
clip_model, preprocess_clip = clip.load("ViT-L/14", device=device)
clip_model = clip_model.float().eval()
clip_base_wrapper = CLIPWrapper(clip_model.visual).eval()

print("Cargando CLIP + LoRA...")
clip_model2, _ = clip.load("ViT-L/14", device=device)
clip_model2 = clip_model2.float()
visual_lora = clip_model2.visual
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["out_proj", "c_fc", "c_proj"], bias="none"
)
visual_lora = get_peft_model(visual_lora, lora_config).to(device)
ckpt = torch.load(CHECKPOINT_CLIP, map_location=device, weights_only=False)
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
visual_lora.load_state_dict(state_dict)
visual_lora.eval()
clip_lora_wrapper = CLIPWrapper(visual_lora).eval()

print("Cargando DINOv2 base...")
dino_base = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14_reg')
dino_base = dino_base.float().to(device).eval()
dino_base_wrapper = DINOWrapper(dino_base).eval()

print("Cargando DINOv2 + LoRA...")
dino_lora_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14_reg')
lora_config_dino = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["qkv", "proj"], bias="none"
)
dino_lora_model = get_peft_model(dino_lora_model, lora_config_dino).to(device)
ckpt_dino = torch.load(CHECKPOINT_DINO, map_location=device, weights_only=False)
state_dict_dino = {k.replace('module.', ''): v for k, v in ckpt_dino['model_state'].items()}
dino_lora_model.load_state_dict(state_dict_dino)
dino_lora_model.eval()
dino_lora_wrapper = DINOWrapper(dino_lora_model).eval()

# ==========================================
# 4. CONFIGURAR GRAD-CAM POR MODELO
# ==========================================
# CLIP base - apuntar a ln_1 del último bloque
target_clip_base = [clip_model.visual.transformer.resblocks[-1].ln_1]
cam_clip_base = GradCAM(
    model=clip_base_wrapper,
    target_layers=target_clip_base,
    reshape_transform=reshape_transform_clip
)

# CLIP + LoRA - apuntar a ln_1 del último bloque (dentro de base_model)
target_clip_lora = [visual_lora.base_model.model.transformer.resblocks[-1].ln_1]
cam_clip_lora = GradCAM(
    model=clip_lora_wrapper,
    target_layers=target_clip_lora,
    reshape_transform=reshape_transform_clip
)

# DINOv2 base - apuntar a norm1 del último bloque
target_dino_base = [dino_base.blocks[-1].norm1]
cam_dino_base = GradCAM(
    model=dino_base_wrapper,
    target_layers=target_dino_base,
    reshape_transform=reshape_transform_dino
)

# DINOv2 + LoRA - apuntar a norm1 del último bloque (dentro de base_model)
target_dino_lora = [dino_lora_model.base_model.model.blocks[-1].norm1]
cam_dino_lora = GradCAM(
    model=dino_lora_wrapper,
    target_layers=target_dino_lora,
    reshape_transform=reshape_transform_dino
)

modelos = [
    ("CLIP base",     cam_clip_base,  preprocess_clip, MEAN_CLIP, STD_CLIP),
    ("CLIP + LoRA",   cam_clip_lora,  preprocess_clip, MEAN_CLIP, STD_CLIP),
    ("DINOv2 base",   cam_dino_base,  None,            MEAN_DINO, STD_DINO),
    ("DINOv2 + LoRA", cam_dino_lora,  None,            MEAN_DINO, STD_DINO),
]

preprocess_dino = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ==========================================
# 5. GENERAR FIGURA COMPARATIVA
# ==========================================
n_imgs    = len(IMAGENES)
n_modelos = len(modelos)

fig, axes = plt.subplots(n_imgs, n_modelos + 1, figsize=(5*(n_modelos+1), 5*n_imgs))
fig.suptitle("Mapas de Atención Grad-CAM - Comparativa modelos en CASIA-WebFace",
             fontsize=16, fontweight='bold', y=1.01)

col_nombres = ["Original"] + [m[0] for m in modelos]
for col, nombre in enumerate(col_nombres):
    axes[0, col].set_title(nombre, fontsize=12, fontweight='bold', pad=10)

for row, ruta_img in enumerate(IMAGENES):
    print(f"Procesando imagen {row+1}/{n_imgs}...")
    img_pil = Image.open(ruta_img).convert('RGB').resize((224, 224))

    # Columna 0 = imagen original
    axes[row, 0].imshow(img_pil)
    axes[row, 0].axis('off')

    for col, (nombre, cam, preprocess, mean, std) in enumerate(modelos):
        try:
            # Preprocesar
            if preprocess is not None:
                input_tensor = preprocess(img_pil).unsqueeze(0).to(device)
            else:
                input_tensor = preprocess_dino(img_pil).unsqueeze(0).to(device)

            input_tensor.requires_grad_(True)

            # Grad-CAM
            with torch.set_grad_enabled(True):
                grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]

            # Desnormalizar para visualización
            img_unnorm = input_tensor[0] * std + mean
            img_unnorm = img_unnorm.permute(1,2,0).detach().cpu().numpy()
            img_unnorm = np.clip(img_unnorm, 0, 1)

            visualizacion = show_cam_on_image(img_unnorm, grayscale_cam, use_rgb=True)
            axes[row, col+1].imshow(visualizacion)

        except Exception as e:
            print(f"  Error en {nombre}: {e}")
            axes[row, col+1].imshow(img_pil)

        axes[row, col+1].axis('off')

plt.tight_layout()
ruta_salida = os.path.join(DIRECTORIO_OUT, "gradcam_comparativa_4modelos.png")
plt.savefig(ruta_salida, dpi=300, bbox_inches='tight')
plt.show()
print(f"\n Gráfica guardada: {ruta_salida}")

# Entrenamiento con CLIP-ViT-Large-14 y LORA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from peft import LoraConfig, get_peft_model
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
DATASET_RUTA     = ""
DATASET_RUTA_VAL = ""
DIRECTORIO_OUT   = ""
NOMBRE_ORIGEN    = "Digiface"
LIMITE           = 5000
LIMITE_VAL       = 1000

LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1

EPOCHS         = 15
PACIENCIA      = 10
DELTA          = 1e-4
BATCH_SIZE     = 64
LR             = 1e-5

OUT_NPY              = os.path.join(DIRECTORIO_OUT, f"embeddings_{NOMBRE_ORIGEN.lower()}_clip_lora_ce.npy")
OUT_CSV              = os.path.join(DIRECTORIO_OUT, f"embeddings_{NOMBRE_ORIGEN.lower()}_clip_lora_ce.csv")
OUT_MODEL            = os.path.join(DIRECTORIO_OUT, "modelo_clip_loraDIGI_ce_best")
OUT_GRAFICAS         = os.path.join(DIRECTORIO_OUT, "training_curvesDIGI_ce.png")
CHECKPOINT_PATH      = os.path.join(DIRECTORIO_OUT, "checkpoint_ultimoDIGI_ce_train.pt")
BEST_MODEL_PATH      = os.path.join(DIRECTORIO_OUT, "checkpoint_bestDIGI_ce_train.pt")

REANUDAR = True

# ==========================================
# 2. CARGA CLIP + LoRA
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

print("Cargando CLIP ViT-L/14...")
base_model, preprocess = clip.load("ViT-L/14", device="cpu")
visual_encoder = base_model.visual
visual_encoder.to(device)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["out_proj", "c_fc", "c_proj"],
    bias="none"
)
visual_encoder = get_peft_model(visual_encoder, lora_config)
visual_encoder.print_trainable_parameters()

if torch.cuda.device_count() > 1:
    print(f"Usando {torch.cuda.device_count()} GPUs T4")
    visual_encoder = nn.DataParallel(visual_encoder)

visual_encoder = visual_encoder.to(device)

# ==========================================
# 3. DATASET
# ==========================================
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform, limite=None):
        self.samples   = []
        self.transform = transform
        self.label2idx = {}
        EXTENSIONES    = {'.jpg', '.jpeg', '.png'}

        for identity in sorted(os.listdir(root_dir)):
            if limite and len(self.samples) >= limite:
                break
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path):
                continue
            if identity not in self.label2idx:
                self.label2idx[identity] = len(self.label2idx)
            for img_name in sorted(os.listdir(id_path)):
                if limite and len(self.samples) >= limite:
                    break
                if os.path.splitext(img_name)[1].lower() in EXTENSIONES:
                    self.samples.append((
                        os.path.join(id_path, img_name),
                        identity,
                        self.label2idx[identity]
                    ))

        self.n_clases = len(self.label2idx)
        print(f"Imágenes: {len(self.samples)} | Identidades: {self.n_clases}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, identity, label = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, torch.tensor(label, dtype=torch.long)

class FaceDatasetVal(Dataset):
    def __init__(self, root_dir, transform, limite=None):
        self.samples   = []
        self.transform = transform
        EXTENSIONES    = {'.jpg', '.jpeg', '.png'}

        for identity in sorted(os.listdir(root_dir)):
            if limite and len(self.samples) >= limite:
                break
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path):
                continue
            for img_name in sorted(os.listdir(id_path)):
                if limite and len(self.samples) >= limite:
                    break
                if os.path.splitext(img_name)[1].lower() in EXTENSIONES:
                    self.samples.append((
                        os.path.join(id_path, img_name),
                        identity
                    ))

        print(f"Val - Imágenes: {len(self.samples)} | Identidades: {len(set(s[1] for s in self.samples))}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, identity = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, identity

# ==========================================
# 4.  PREPARAR EL ENTRENAMIENTO
# ==========================================
print("\nCargando dataset train...")
dataset     = FaceDataset(DATASET_RUTA, preprocess, limite=LIMITE)
loader      = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)

print("Cargando dataset validación (CASIA-WebFace)...")
dataset_val = FaceDatasetVal(DATASET_RUTA_VAL, preprocess, limite=LIMITE_VAL)
loader_val  = DataLoader(dataset_val, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

classifier = nn.Linear(768, dataset.n_clases).to(device)
criterion  = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {'params': filter(lambda p: p.requires_grad, visual_encoder.parameters()), 'lr': LR},
    {'params': classifier.parameters(), 'lr': LR * 10}
])

scaler    = torch.amp.GradScaler('cuda')
history   = {'loss': [], 'val_gap': [], 'val_gen': [], 'val_imp': []}
START_EPOCH = 0

# ==========================================
# REANUDAR
# ==========================================
if REANUDAR and os.path.exists(CHECKPOINT_PATH):
    print(f"Reanudando desde: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device,weights_only=False)
    visual_encoder.load_state_dict(ckpt['model_state'])
    classifier.load_state_dict(ckpt['classifier_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    history     = ckpt['history']
    START_EPOCH = ckpt['epoch'] + 1
    mejor_loss  = min(history['loss'])
    print(f"Continuando desde epoch {START_EPOCH + 1} | Mejor loss: {mejor_loss:.4f}")
else:
    print("Entrenamiento desde cero")

# ==========================================
#  EARLY STOPPING
# ==========================================
class EarlyStopping:
    def __init__(self, paciencia=5, delta=1e-4):
        self.paciencia  = paciencia
        self.delta      = delta
        self.mejor_val  = -float('inf')  # mayor gap = mejor
        self.contador   = 0
        self.parar      = False

    def step(self, gap):
        if gap > self.mejor_val + self.delta:
            self.mejor_val = gap
            self.contador  = 0
        else:
            self.contador += 1
            if self.contador >= self.paciencia:
                self.parar = True

early_stopping = EarlyStopping(paciencia=PACIENCIA, delta=DELTA)
mejor_val_gap  = -float('inf')

# ==========================================
# ENTRENAMIENTO
# ==========================================
print(f"\nIniciando entrenamiento CE - máx {EPOCHS} epochs | Paciencia: {PACIENCIA}")
print(f"Train: {len(dataset)} imágenes | Val: {len(dataset_val)} imágenes\n")

for epoch in range(START_EPOCH, EPOCHS):

    # --- TRAIN ---
    visual_encoder.train()
    classifier.train()
    total_loss = 0.0

    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1} [Train]"):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            z = visual_encoder(imgs)
            if not isinstance(z, torch.Tensor):
                z = z[0]
            embeddings = F.normalize(z.float(), dim=-1)
            logits     = classifier(embeddings)
            loss       = criterion(logits, labels)
            scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    history['loss'].append(avg_loss)

    visual_encoder.eval()
    embs_val = []
    ids_val  = []

    with torch.no_grad():
        for imgs, ids in tqdm(loader_val, desc=f"Epoch {epoch+1} [Val]"):
            imgs = imgs.to(device)
            with torch.amp.autocast('cuda'):
                z = visual_encoder(imgs)
                if not isinstance(z, torch.Tensor):
                    z = z[0]
            z = F.normalize(z.float(), dim=-1)
            embs_val.append(z.cpu().numpy())
            ids_val.extend(ids)

    embs_val = np.vstack(embs_val)
    ids_arr  = np.array(ids_val)
    ids_unicas = np.unique(ids_arr)
    ids_validas = [i for i in ids_unicas if (ids_arr == i).sum() >= 2]

    rng = np.random.default_rng(42)
    scores_gen, scores_imp = [], []

    for _ in range(300):
        ide = rng.choice(ids_validas)
        idx = np.where(ids_arr == ide)[0]
        i, j = rng.choice(idx, 2, replace=False)
        scores_gen.append(np.dot(embs_val[i], embs_val[j]))

    for _ in range(300):
        id1, id2 = rng.choice(ids_validas, 2, replace=False)
        i = rng.choice(np.where(ids_arr == id1)[0])
        j = rng.choice(np.where(ids_arr == id2)[0])
        scores_imp.append(np.dot(embs_val[i], embs_val[j]))

    scores_gen = np.array(scores_gen)
    scores_imp = np.array(scores_imp)
    gap        = scores_gen.mean() - scores_imp.mean()

    history['val_gap'].append(gap)
    history['val_gen'].append(scores_gen.mean())
    history['val_imp'].append(scores_imp.mean())

    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"Epoch {epoch+1} - Train Loss: {avg_loss:.4f} - Val Gap: {gap:.4f} "
          f"(gen: {scores_gen.mean():.4f} | imp: {scores_imp.mean():.4f}) - VRAM: {vram:.2f} GB")

    # Checkpoint
    torch.save({
        'epoch':            epoch,
        'model_state':      visual_encoder.state_dict(),
        'classifier_state': classifier.state_dict(),
        'optimizer_state':  optimizer.state_dict(),
        'scaler_state':     scaler.state_dict(),
        'history':          history
    }, CHECKPOINT_PATH)

    #  Mejor modelo 
    if gap > mejor_val_gap:
        mejor_val_gap = gap
        torch.save({
            'epoch':            epoch,
            'model_state':      visual_encoder.state_dict(),
            'classifier_state': classifier.state_dict(),
            'history':          history
        }, BEST_MODEL_PATH)
        print(f" Mejor modelo guardado - Val Gap: {mejor_val_gap:.4f}")

    #  Early Stopping 
    early_stopping.step(gap)
    if early_stopping.parar:
        print(f"\ Early Stopping en epoch {epoch+1}")
        print(f"   Mejor Val Gap: {early_stopping.mejor_val:.4f}")
        break
    else:
        if gap <= mejor_val_gap:
            print(f" Sin mejora val: {early_stopping.contador}/{PACIENCIA}")

# ==========================================
#  GRÁFICAS
# ==========================================
epochs_range = range(1, len(history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Curvas de Entrenamiento - CLIP LoRA + CE", fontsize=14, fontweight='bold')

# Train Loss
axes[0].plot(epochs_range, history['loss'], color='steelblue', linewidth=2, marker='o', markersize=4)
axes[0].set_title("Train Loss (IDiffFace)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss CE")
axes[0].grid(True, alpha=0.3)

# Val Gap
axes[1].plot(epochs_range, history['val_gap'], color='darkorange',  linewidth=2, marker='o', markersize=4, label='Gap')
axes[1].plot(epochs_range, history['val_gen'], color='green',       linewidth=1, linestyle='--', label='Genuinos')
axes[1].plot(epochs_range, history['val_imp'], color='red',         linewidth=1, linestyle='--', label='Impostores')
axes[1].set_title("Gap Genuinos/Impostores en CASIA-WebFace")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Similitud coseno")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_GRAFICAS, dpi=150, bbox_inches='tight')
plt.show()
print(f"Gráficas guardadas: {OUT_GRAFICAS}")

# ==========================================
# CARGAR MEJOR MODELO Y EXTRAER EMBEDDINGS
# ==========================================
print("\nCargando mejor modelo para extracción...")
best_ckpt = torch.load(BEST_MODEL_PATH, map_location=device,weights_only=False)
visual_encoder.load_state_dict(best_ckpt['model_state'])
visual_encoder.module.save_pretrained(OUT_MODEL)
print(f"Mejor modelo LoRA guardado: {OUT_MODEL}")

print("\nExtrayendo embeddings...")
extract_loader = DataLoader(dataset, batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=True)

visual_encoder.eval()
embeddings_list = []

with torch.no_grad():
    for imgs, _ in tqdm(extract_loader, desc="Extrayendo"):
        with torch.amp.autocast('cuda'):
            z = visual_encoder(imgs.to(device))
            if not isinstance(z, torch.Tensor):
                z = z[0]
        z = F.normalize(z.float(), dim=-1)
        embeddings_list.append(z.cpu().numpy())

embeddings_array = np.vstack(embeddings_list)
np.save(OUT_NPY, embeddings_array)

df = pd.DataFrame({
    'img_path':          [s[0] for s in dataset.samples],
    'img_path_relativo': [os.path.relpath(s[0], DATASET_RUTA) for s in dataset.samples],
    'identity':          [s[1] for s in dataset.samples],
    'origen':            NOMBRE_ORIGEN,
    'modelo':            'clip_lora_ce'
})
df.to_csv(OUT_CSV, index=False)

assert len(embeddings_array) == len(df), \
    f"Desajuste emb:{len(embeddings_array)} csv:{len(df)}"

print(f"\ Embeddings: {embeddings_array.shape} EN {OUT_NPY}")
print(f" CSV: {OUT_CSV}")
print("\nCOMPLETADO")

# Entrenamiento con DINOv2 y LORA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from peft import LoraConfig, get_peft_model
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
DATASET_RUTA     = ""
DATASET_RUTA_VAL = ""
DIRECTORIO_OUT   = ""
NOMBRE_ORIGEN    = "IDIFF-FACE-U"
LIMITE           = 5000
LIMITE_VAL       = 1000

LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1

EPOCHS         = 80
PACIENCIA      = 10
DELTA          = 1e-4
BATCH_SIZE     = 64
LR             = 1e-5

OUT_NPY              = os.path.join(DIRECTORIO_OUT, f"embeddings_{NOMBRE_ORIGEN.lower()}_dino_lora_ce.npy")
OUT_CSV              = os.path.join(DIRECTORIO_OUT, f"embeddings_{NOMBRE_ORIGEN.lower()}_dino_lora_ce.csv")
OUT_MODEL            = os.path.join(DIRECTORIO_OUT, "modelo_dino_loraIDIFF_ce_best")
OUT_GRAFICAS         = os.path.join(DIRECTORIO_OUT, "training_curves_dinoIDIFF_ce.png")
CHECKPOINT_PATH      = os.path.join(DIRECTORIO_OUT, "checkpoint_ultimo_dinoIDIFF_ce_train.pt")
BEST_MODEL_PATH      = os.path.join(DIRECTORIO_OUT, "checkpoint_best_dino_ceIDIFF_train.pt")

REANUDAR = False

# ==========================================
# 2. CARGA DINOv2 + LoRA
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

print("Cargando DINOv2 ViT-L/14 reg...")
visual_encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14_reg')
visual_encoder.to(device)

# Preprocesado estándar DINOv2
preprocess = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# LoRA - capas de atención accesibles en DINOv2
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["qkv", "proj"],  # capas de atención en DINOv2
    bias="none"
)
visual_encoder = get_peft_model(visual_encoder, lora_config)
visual_encoder.print_trainable_parameters()

if torch.cuda.device_count() > 1:
    print(f"Usando {torch.cuda.device_count()} GPUs T4")
    visual_encoder = nn.DataParallel(visual_encoder)

visual_encoder = visual_encoder.to(device)
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ==========================================
# 3. DATASET
# ==========================================
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform, limite=None):
        self.samples   = []
        self.transform = transform
        self.label2idx = {}
        EXTENSIONES    = {'.jpg', '.jpeg', '.png'}

        for identity in sorted(os.listdir(root_dir)):
            if limite and len(self.samples) >= limite:
                break
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path):
                continue
            if identity not in self.label2idx:
                self.label2idx[identity] = len(self.label2idx)
            for img_name in sorted(os.listdir(id_path)):
                if limite and len(self.samples) >= limite:
                    break
                if os.path.splitext(img_name)[1].lower() in EXTENSIONES:
                    self.samples.append((
                        os.path.join(id_path, img_name),
                        identity,
                        self.label2idx[identity]
                    ))

        self.n_clases = len(self.label2idx)
        print(f"Imágenes: {len(self.samples)} | Identidades: {self.n_clases}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, identity, label = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, torch.tensor(label, dtype=torch.long)

class FaceDatasetVal(Dataset):
    def __init__(self, root_dir, transform, limite=None):
        self.samples   = []
        self.transform = transform
        EXTENSIONES    = {'.jpg', '.jpeg', '.png'}

        for identity in sorted(os.listdir(root_dir)):
            if limite and len(self.samples) >= limite:
                break
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path):
                continue
            for img_name in sorted(os.listdir(id_path)):
                if limite and len(self.samples) >= limite:
                    break
                if os.path.splitext(img_name)[1].lower() in EXTENSIONES:
                    self.samples.append((
                        os.path.join(id_path, img_name),
                        identity
                    ))

        print(f"Val - Imágenes: {len(self.samples)} | Identidades: {len(set(s[1] for s in self.samples))}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, identity = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, identity

# ==========================================
# 4. ENTRENAMIENTO
# ==========================================
print("\nCargando dataset train...")
dataset     = FaceDataset(DATASET_RUTA, preprocess, limite=LIMITE)
loader      = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)

print("Cargando dataset validación (CASIA-WebFace)...")
dataset_val = FaceDatasetVal(DATASET_RUTA_VAL, preprocess, limite=LIMITE_VAL)
loader_val  = DataLoader(dataset_val, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

# DINOv2 ViT-L/14 tiene embedding de 1024 
classifier = nn.Linear(1024, dataset.n_clases).to(device)
criterion  = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {'params': filter(lambda p: p.requires_grad, visual_encoder.parameters()), 'lr': LR},
    {'params': classifier.parameters(), 'lr': LR * 10}
])

scaler    = torch.amp.GradScaler('cuda')
history   = {'loss': [], 'val_gap': [], 'val_gen': [], 'val_imp': []}
START_EPOCH = 0

# ==========================================
#  REANUDAR
# ==========================================
if REANUDAR and os.path.exists(CHECKPOINT_PATH):
    print(f"Reanudando desde: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    visual_encoder.load_state_dict(ckpt['model_state'])
    classifier.load_state_dict(ckpt['classifier_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    history     = ckpt['history']
    START_EPOCH = ckpt['epoch'] + 1
    mejor_val_gap = max(history['val_gap']) if history['val_gap'] else -float('inf')
    early_stopping_mejor = mejor_val_gap
    print(f"Continuando desde epoch {START_EPOCH + 1} | Mejor val_gap: {mejor_val_gap:.4f}")
else:
    print("Entrenamiento desde cero")

# ==========================================
#  EARLY STOPPING
# ==========================================
class EarlyStopping:
    def __init__(self, paciencia=5, delta=1e-4):
        self.paciencia  = paciencia
        self.delta      = delta
        self.mejor_val  = -float('inf')
        self.contador   = 0
        self.parar      = False

    def step(self, gap):
        if gap > self.mejor_val + self.delta:
            self.mejor_val = gap
            self.contador  = 0
        else:
            self.contador += 1
            if self.contador >= self.paciencia:
                self.parar = True

early_stopping = EarlyStopping(paciencia=PACIENCIA, delta=DELTA)
mejor_val_gap  = -float('inf')

# ==========================================
# LOOP DE ENTRENAMIENTO
# ==========================================
print(f"\nIniciando entrenamiento CE - máx {EPOCHS} epochs | Paciencia: {PACIENCIA}")
print(f"Train: {len(dataset)} imágenes | Val: {len(dataset_val)} imágenes\n")

for epoch in range(START_EPOCH, EPOCHS):

    # --- TRAIN ---
    visual_encoder.train()
    classifier.train()
    total_loss = 0.0

    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1} [Train]"):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            z = visual_encoder(imgs)
            if not isinstance(z, torch.Tensor):
                z = z[0]
            embeddings = F.normalize(z.float(), dim=-1)
            logits     = classifier(embeddings)
            loss       = criterion(logits, labels)
            scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    history['loss'].append(avg_loss)

    # --- VALIDACIÓN ---
    visual_encoder.eval()
    embs_val = []
    ids_val  = []

    with torch.no_grad():
        for imgs, ids in tqdm(loader_val, desc=f"Epoch {epoch+1} [Val]"):
            imgs = imgs.to(device)
            with torch.amp.autocast('cuda'):
                z = visual_encoder(imgs)
                if not isinstance(z, torch.Tensor):
                    z = z[0]
            z = F.normalize(z.float(), dim=-1)
            embs_val.append(z.cpu().numpy())
            ids_val.extend(ids)

    embs_val    = np.vstack(embs_val)
    ids_arr     = np.array(ids_val)
    ids_unicas  = np.unique(ids_arr)
    ids_validas = [i for i in ids_unicas if (ids_arr == i).sum() >= 2]

    rng = np.random.default_rng(42)
    scores_gen, scores_imp = [], []

    for _ in range(300):
        ide = rng.choice(ids_validas)
        idx = np.where(ids_arr == ide)[0]
        i, j = rng.choice(idx, 2, replace=False)
        scores_gen.append(np.dot(embs_val[i], embs_val[j]))

    for _ in range(300):
        id1, id2 = rng.choice(ids_validas, 2, replace=False)
        i = rng.choice(np.where(ids_arr == id1)[0])
        j = rng.choice(np.where(ids_arr == id2)[0])
        scores_imp.append(np.dot(embs_val[i], embs_val[j]))

    scores_gen = np.array(scores_gen)
    scores_imp = np.array(scores_imp)
    gap        = scores_gen.mean() - scores_imp.mean()

    history['val_gap'].append(gap)
    history['val_gen'].append(scores_gen.mean())
    history['val_imp'].append(scores_imp.mean())

    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"Epoch {epoch+1} - Train Loss: {avg_loss:.4f} - Val Gap: {gap:.4f} "
          f"(gen: {scores_gen.mean():.4f} | imp: {scores_imp.mean():.4f}) - VRAM: {vram:.2f} GB")

    # Checkpoint
    torch.save({
        'epoch':            epoch,
        'model_state':      visual_encoder.state_dict(),
        'classifier_state': classifier.state_dict(),
        'optimizer_state':  optimizer.state_dict(),
        'scaler_state':     scaler.state_dict(),
        'history':          history
    }, CHECKPOINT_PATH)

    #  Mejor modelo
    if gap > mejor_val_gap:
        mejor_val_gap = gap
        torch.save({
            'epoch':            epoch,
            'model_state':      visual_encoder.state_dict(),
            'classifier_state': classifier.state_dict(),
            'history':          history
        }, BEST_MODEL_PATH)
        print(f" Mejor modelo guardado - Val Gap: {mejor_val_gap:.4f}")

    #  Stopping
    early_stopping.step(gap)
    if early_stopping.parar:
        print(f"\ Early Stopping en epoch {epoch+1}")
        print(f"   Mejor Val Gap: {early_stopping.mejor_val:.4f}")
        break
    else:
        if gap <= mejor_val_gap:
            print(f" Sin mejora val: {early_stopping.contador}/{PACIENCIA}")

# ==========================================
#  GRÁFICAS
# ==========================================
epochs_range = range(1, len(history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Curvas de Entrenamiento - DINOv2 LoRA + CE", fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history['loss'], color='steelblue', linewidth=2, marker='o', markersize=4)
axes[0].set_title("Train Loss (IDiffFace)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss CE")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_gap'], color='darkorange', linewidth=2, marker='o', markersize=4, label='Gap')
axes[1].plot(epochs_range, history['val_gen'], color='green',      linewidth=1, linestyle='--', label='Genuinos')
axes[1].plot(epochs_range, history['val_imp'], color='red',        linewidth=1, linestyle='--', label='Impostores')
axes[1].set_title("Gap Genuinos/Impostores en CASIA-WebFace")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Similitud coseno")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_GRAFICAS, dpi=150, bbox_inches='tight')
plt.show()
print(f"Gráficas guardadas: {OUT_GRAFICAS}")

# ==========================================
# CARGAR MEJOR MODELO Y EXTRAER EMBEDDINGS
# ==========================================
print("\nCargando mejor modelo para extracción...")
best_ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
visual_encoder.load_state_dict(best_ckpt['model_state'])
visual_encoder.module.save_pretrained(OUT_MODEL)
print(f"Mejor modelo LoRA guardado: {OUT_MODEL}")

print("\nExtrayendo embeddings...")
extract_loader = DataLoader(dataset, batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=True)

visual_encoder.eval()
embeddings_list = []

with torch.no_grad():
    for imgs, _ in tqdm(extract_loader, desc="Extrayendo"):
        with torch.amp.autocast('cuda'):
            z = visual_encoder(imgs.to(device))
            if not isinstance(z, torch.Tensor):
                z = z[0]
        z = F.normalize(z.float(), dim=-1)
        embeddings_list.append(z.cpu().numpy())

embeddings_array = np.vstack(embeddings_list)
np.save(OUT_NPY, embeddings_array)

df = pd.DataFrame({
    'img_path':          [s[0] for s in dataset.samples],
    'img_path_relativo': [os.path.relpath(s[0], DATASET_RUTA) for s in dataset.samples],
    'identity':          [s[1] for s in dataset.samples],
    'origen':            NOMBRE_ORIGEN,
    'modelo':            'dino_lora_ce'
})
df.to_csv(OUT_CSV, index=False)

assert len(embeddings_array) == len(df), \
    f"Desajuste emb:{len(embeddings_array)} csv:{len(df)}"

print(f"\n Embeddings: {embeddings_array.shape} en {OUT_NPY}")
print(f" CSV: {OUT_CSV}")
print("\nCOMPLETADO")

# Entrenamiento RESNET-50 y FULL FINE TUNNING

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
DATASET_RUTA_VAL = ""
DIRECTORIO_OUT   = ""

EPOCHS     = 80
PACIENCIA  = 10
DELTA      = 1e-4
BATCH_SIZE = 128
LR         = 1e-4
LIMITE_VAL = 10000

# Datasets sobre los que entrenar en bucle
EXPERIMENTOS = [
    {
        "nombre":        "DigiFace",
        "dataset_ruta":  "",
        "limite":        25000,
        "out_npy":       os.path.join(DIRECTORIO_OUT, "embeddings_digiface_resnet50_ft.npy"),
        "out_csv":       os.path.join(DIRECTORIO_OUT, "embeddings_digiface_resnet50_ft.csv"),
        "out_graficas":  os.path.join(DIRECTORIO_OUT, "training_curves_resnet50_digiface.png"),
        "checkpoint":    os.path.join(DIRECTORIO_OUT, "checkpoint_resnet50_digiface_last.pt"),
        "best_model":    os.path.join(DIRECTORIO_OUT, "checkpoint_resnet50_digiface_best.pt"),
    },
    {
        "nombre":        "IDiffFace-U",
        "dataset_ruta":  "",
        "limite":        25000,
        "out_npy":       os.path.join(DIRECTORIO_OUT, "embeddings_idiff_resnet50_ft.npy"),
        "out_csv":       os.path.join(DIRECTORIO_OUT, "embeddings_idiff_resnet50_ft.csv"),
        "out_graficas":  os.path.join(DIRECTORIO_OUT, "training_curves_resnet50_idiff.png"),
        "checkpoint":    os.path.join(DIRECTORIO_OUT, "checkpoint_resnet50_idiff_last.pt"),
        "best_model":    os.path.join(DIRECTORIO_OUT, "checkpoint_resnet50_idiff_best.pt"),
    },
]

# Preprocesado estándar ResNet50
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. DATASETS
# ==========================================
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform, limite=None):
        self.samples   = []
        self.transform = transform
        self.label2idx = {}
        EXTENSIONES    = {'.jpg', '.jpeg', '.png'}

        for identity in sorted(os.listdir(root_dir)):
            if limite and len(self.samples) >= limite:
                break
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path):
                continue
            if identity not in self.label2idx:
                self.label2idx[identity] = len(self.label2idx)
            for img_name in sorted(os.listdir(id_path)):
                if limite and len(self.samples) >= limite:
                    break
                if os.path.splitext(img_name)[1].lower() in EXTENSIONES:
                    self.samples.append((
                        os.path.join(id_path, img_name),
                        identity,
                        self.label2idx[identity]
                    ))

        self.n_clases = len(self.label2idx)
        print(f"Imágenes: {len(self.samples)} | Identidades: {self.n_clases}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, identity, label = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, torch.tensor(label, dtype=torch.long)


class FaceDatasetVal(Dataset):
    def __init__(self, root_dir, transform, limite=None):
        self.samples   = []
        self.transform = transform
        EXTENSIONES    = {'.jpg', '.jpeg', '.png'}

        for identity in sorted(os.listdir(root_dir)):
            if limite and len(self.samples) >= limite:
                break
            id_path = os.path.join(root_dir, identity)
            if not os.path.isdir(id_path):
                continue
            for img_name in sorted(os.listdir(id_path)):
                if limite and len(self.samples) >= limite:
                    break
                if os.path.splitext(img_name)[1].lower() in EXTENSIONES:
                    self.samples.append((
                        os.path.join(id_path, img_name),
                        identity
                    ))

        print(f"Val - Imágenes: {len(self.samples)} | Identidades: {len(set(s[1] for s in self.samples))}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, identity = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, identity


# ==========================================
# 3. FUNCIÓN DE CONSTRUCCIÓN DEL MODELO
# ==========================================
def construir_resnet50(n_clases, device):
    """ResNet50 preentrenado en ImageNet con cabeza clasificadora sustituida"""
    backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Sustituir la cabeza clasificadora por identidad - extraeremos embeddings antes
    embedding_dim = backbone.fc.in_features  # 2048
    backbone.fc   = nn.Identity()            # sin clasificación

    if torch.cuda.device_count() > 1:
        print(f"Usando {torch.cuda.device_count()} GPUs T4")
        backbone = nn.DataParallel(backbone)

    backbone = backbone.to(device)

    # Clasificador separado sobre los 2048 dims del embedding
    classifier = nn.Linear(embedding_dim, n_clases).to(device)

    total     = sum(p.numel() for p in backbone.parameters())
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f"ResNet50 - Parámetros totales: {total:,} | Entrenables: {trainable:,} ({100*trainable/total:.1f}%)")

    return backbone, classifier, embedding_dim


# ==========================================
# 4. FUNCIÓN DE ENTRENAMIENTO
# ==========================================
def entrenar(exp, dataset_val, loader_val, device):
    nombre       = exp['nombre']
    dataset_ruta = exp['dataset_ruta']
    limite       = exp['limite']

    print(f"\n{'='*60}")
    print(f"EXPERIMENTO: ResNet50 fine-tuning - {nombre}")
    print(f"{'='*60}")

    # Dataset train
    print(f"\nCargando dataset train ({nombre})...")
    dataset = FaceDataset(dataset_ruta, resnet_transform, limite=limite)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)

    # Modelo
    backbone, classifier, embedding_dim = construir_resnet50(dataset.n_clases, device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([
        {'params': backbone.parameters(),    'lr': LR,      'weight_decay': 1e-4},
        {'params': classifier.parameters(),  'lr': LR * 10, 'weight_decay': 1e-4}
    ])
    scaler = torch.amp.GradScaler('cuda')

    history        = {'loss': [], 'val_gap': [], 'val_gen': [], 'val_imp': []}
    mejor_val_gap  = -float('inf')

    # Early Stopping
    class EarlyStopping:
        def __init__(self, paciencia=10, delta=1e-4):
            self.paciencia = paciencia
            self.delta     = delta
            self.mejor_val = -float('inf')
            self.contador  = 0
            self.parar     = False

        def step(self, gap):
            if gap > self.mejor_val + self.delta:
                self.mejor_val = gap
                self.contador  = 0
            else:
                self.contador += 1
                if self.contador >= self.paciencia:
                    self.parar = True

    early_stopping = EarlyStopping(paciencia=PACIENCIA, delta=DELTA)

    print(f"\nIniciando entrenamiento - máx {EPOCHS} epochs | Paciencia: {PACIENCIA}")
    print(f"Train: {len(dataset)} imgs | Val: {len(dataset_val)} imgs\n")

    for epoch in range(EPOCHS):

        # --- TRAIN ---
        backbone.train()
        classifier.train()
        total_loss = 0.0

        for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1} [Train]"):
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                z          = backbone(imgs)
                embeddings = F.normalize(z.float(), dim=-1)
                logits     = classifier(embeddings)
                loss       = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        history['loss'].append(avg_loss)

        # --- VALIDACIÓN ---
        backbone.eval()
        embs_val = []
        ids_val  = []

        with torch.no_grad():
            for imgs, ids in tqdm(loader_val, desc=f"Epoch {epoch+1} [Val]"):
                imgs = imgs.to(device)
                with torch.amp.autocast('cuda'):
                    z = backbone(imgs)
                z = F.normalize(z.float(), dim=-1)
                embs_val.append(z.cpu().numpy())
                ids_val.extend(ids)

        embs_val    = np.vstack(embs_val)
        ids_arr     = np.array(ids_val)
        ids_unicas  = np.unique(ids_arr)
        ids_validas = [i for i in ids_unicas if (ids_arr == i).sum() >= 2]

        rng = np.random.default_rng(42)
        scores_gen, scores_imp = [], []

        for _ in range(300):
            ide = rng.choice(ids_validas)
            idx = np.where(ids_arr == ide)[0]
            i, j = rng.choice(idx, 2, replace=False)
            scores_gen.append(np.dot(embs_val[i], embs_val[j]))

        for _ in range(300):
            id1, id2 = rng.choice(ids_validas, 2, replace=False)
            i = rng.choice(np.where(ids_arr == id1)[0])
            j = rng.choice(np.where(ids_arr == id2)[0])
            scores_imp.append(np.dot(embs_val[i], embs_val[j]))

        scores_gen = np.array(scores_gen)
        scores_imp = np.array(scores_imp)
        gap        = scores_gen.mean() - scores_imp.mean()

        history['val_gap'].append(gap)
        history['val_gen'].append(scores_gen.mean())
        history['val_imp'].append(scores_imp.mean())

        vram = torch.cuda.memory_allocated() / 1024**3
        print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f} - Val Gap: {gap:.4f} "
              f"(gen: {scores_gen.mean():.4f} | imp: {scores_imp.mean():.4f}) - VRAM: {vram:.2f} GB")

        # Checkpoint último
        torch.save({
            'epoch':            epoch,
            'model_state':      backbone.module.state_dict() if isinstance(backbone, nn.DataParallel) else backbone.state_dict(),
            'classifier_state': classifier.state_dict(),
            'optimizer_state':  optimizer.state_dict(),
            'scaler_state':     scaler.state_dict(),
            'history':          history
        }, exp['checkpoint'])

        # Mejor modelo
        if gap > mejor_val_gap:
            mejor_val_gap = gap
            torch.save({
                'epoch':            epoch,
                'model_state':      backbone.module.state_dict() if isinstance(backbone, nn.DataParallel) else backbone.state_dict(),
                'classifier_state': classifier.state_dict(),
                'history':          history
            }, exp['best_model'])
            print(f" Mejor modelo guardado - Val Gap: {mejor_val_gap:.4f}")

        early_stopping.step(gap)
        if early_stopping.parar:
            print(f"\n Early Stopping en epoch {epoch+1}")
            print(f"   Mejor Val Gap: {early_stopping.mejor_val:.4f}")
            break
        else:
            if gap <= mejor_val_gap:
                print(f" Sin mejora val: {early_stopping.contador}/{PACIENCIA}")

    return backbone, classifier, dataset, history


# ==========================================
# 5. FUNCIÓN DE GRÁFICAS
# ==========================================
def guardar_graficas(history, nombre, ruta_salida):
    epochs_range = range(1, len(history['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Curvas de Entrenamiento - ResNet50 [{nombre}]",
                 fontsize=14, fontweight='bold')

    axes[0].plot(epochs_range, history['loss'], color='steelblue',
                 linewidth=2, marker='o', markersize=4)
    axes[0].set_title(f"Train Loss ({nombre})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss CE")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history['val_gap'], color='darkorange',
                 linewidth=2, marker='o', markersize=4, label='Gap')
    axes[1].plot(epochs_range, history['val_gen'], color='green',
                 linewidth=1, linestyle='--', label='Genuinos')
    axes[1].plot(epochs_range, history['val_imp'], color='red',
                 linewidth=1, linestyle='--', label='Impostores')
    axes[1].set_title("Gap Genuinos/Impostores en CASIA-WebFace")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Similitud coseno")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(ruta_salida, dpi=150, bbox_inches='tight')
    plt.close()
    print(f" Gráficas guardadas: {ruta_salida}")


# ==========================================
# 6. FUNCIÓN DE EXTRACCIÓN DE EMBEDDINGS
# ==========================================
def extraer_embeddings(backbone, dataset, exp, device):
    print(f"\nExtrayendo embeddings - {exp['nombre']}...")

    # Cargar mejor modelo
    best_ckpt = torch.load(exp['best_model'], map_location=device, weights_only=False)
    if isinstance(backbone, nn.DataParallel):
        backbone.module.load_state_dict(best_ckpt['model_state'])
    else:
        backbone.load_state_dict(best_ckpt['model_state'])
    backbone.eval()

    extract_loader = DataLoader(dataset, batch_size=64, shuffle=False,
                                num_workers=2, pin_memory=True)
    embeddings_list = []

    with torch.no_grad():
        for imgs, _ in tqdm(extract_loader, desc="Extrayendo"):
            with torch.amp.autocast('cuda'):
                z = backbone(imgs.to(device))
            z = F.normalize(z.float(), dim=-1)
            embeddings_list.append(z.cpu().numpy())

    embeddings_array = np.vstack(embeddings_list)
    np.save(exp['out_npy'], embeddings_array)

    df = pd.DataFrame({
        'img_path':          [s[0] for s in dataset.samples],
        'img_path_relativo': [os.path.relpath(s[0], exp['dataset_ruta']) for s in dataset.samples],
        'identity':          [s[1] for s in dataset.samples],
        'origen':            exp['nombre'],
        'modelo':            'resnet50_ft'
    })
    df.to_csv(exp['out_csv'], index=False)

    assert len(embeddings_array) == len(df), \
        f"Desajuste emb:{len(embeddings_array)} csv:{len(df)}"

    print(f" Embeddings: {embeddings_array.shape} en {exp['out_npy']}")
    print(f" CSV: {exp['out_csv']}")


# ==========================================
# 7. BUCLE PRINCIPAL
# ==========================================
print("Cargando dataset de validación (CASIA-WebFace)...")
dataset_val = FaceDatasetVal(DATASET_RUTA_VAL, resnet_transform, limite=LIMITE_VAL)
loader_val  = DataLoader(dataset_val, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

for exp in EXPERIMENTOS:
    torch.cuda.empty_cache()

    # Entrenar
    backbone, classifier, dataset, history = entrenar(
        exp, dataset_val, loader_val, device
    )

    # Gráficas
    guardar_graficas(history, exp['nombre'], exp['out_graficas'])

    # Extraer embeddings
    extraer_embeddings(backbone, dataset, exp, device)

    print(f"\n EXPERIMENTO {exp['nombre']} COMPLETADO\n")
    del backbone, classifier, dataset
    torch.cuda.empty_cache()
